# Lecture 6: Price Elasticity of Demand
## Estimating How Prices Affect Walmart Store Sales

---

### Learning Objectives

By completing this notebook you will be able to:

1. Explore and visualise the relationship between prices, macroeconomic factors, and sales
2. Build log-log regression models and interpret coefficients as elasticities
3. Perform backward variable selection using statistical significance
4. Diagnose regression problems: multicollinearity (VIF) and autocorrelation (Durbin-Watson)
5. Use store fixed effects to control for unobserved store heterogeneity
6. Interpret the business implications of elasticity estimates
7. Apply Ridge and Lasso regularisation to handle multicollinearity and compare their impact on elasticity estimates

### Dataset

We use the **Walmart Weekly Sales** dataset with 6,435 store-week observations across 45 stores:

| Variable | Description |
|----------|-------------|
| `Store` | Store number (1-45) |
| `Date` | Week of sales |
| `Weekly_Sales` | Total weekly sales revenue (\$) |
| `Holiday_Flag` | Whether the week includes a holiday (1 = Yes, 0 = No) |
| `Temperature` | Average temperature in the region (°F) |
| `Fuel_Price` | Cost of fuel in the region (\$/gallon) |
| `CPI` | Consumer Price Index — proxy for general price level |
| `Unemployment` | Regional unemployment rate (%) |

---
## 0. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

DATA_PATH = '/content/Walmart.csv'  # Colab upload location

---
## 1. Load and Explore the Data

### Task 1.1: Load the dataset and inspect its structure

Load `Walmart.csv` into a DataFrame. Print its shape, column types, and summary statistics.

In [2]:
data = pd.read_csv(DATA_PATH, parse_dates=['Date'], dayfirst=True)
print(data.shape); print(data.dtypes); data.describe()

(6435, 8)
Store                    int64
Date            datetime64[us]
Weekly_Sales           float64
Holiday_Flag             int64
Temperature            float64
Fuel_Price             float64
CPI                    float64
Unemployment           float64
dtype: object


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,6435.000000,6435,6.435000e+03,6435.000000,6435.000000,6435.000000,6435.000000,6435.000000
mean,23.000000,2011-06-17 00:00:00,1.046965e+06,0.069930,60.663782,3.358607,171.578394,7.999151
min,1.000000,2010-02-05 00:00:00,2.099862e+05,0.000000,-2.060000,2.472000,126.064000,3.879000
25%,12.000000,2010-10-08 00:00:00,5.533501e+05,0.000000,47.460000,2.933000,131.735000,6.891000
50%,23.000000,2011-06-17 00:00:00,9.607460e+05,0.000000,62.670000,3.445000,182.616521,7.874000
75%,34.000000,2012-02-24 00:00:00,1.420159e+06,0.000000,74.940000,3.735000,212.743293,8.622000
max,45.000000,2012-10-26 00:00:00,3.818686e+06,1.000000,100.140000,4.468000,227.232807,14.313000
std,12.988182,NaN,5.643666e+05,0.255049,18.444933,0.459020,39.356712,1.875885


### Task 1.2: Create log-transformed variables

Create two new columns: `log_CPI` and `log_Weekly_Sales` using the natural log.

**Why logs?** In a log-log model, the coefficient on `ln(CPI)` is directly interpretable as an elasticity:

$$\ln(\text{Weekly\_Sales}) = \beta_0 + \beta_1 \ln(\text{CPI}) + \ldots$$

Here $\beta_1$ represents: *a 1% increase in CPI is associated with a $\beta_1$% change in weekly sales*.

In [3]:
data['log_CPI'], data['log_Weekly_Sales'] = np.log(data.CPI), np.log(data.Weekly_Sales)

### Task 1.3: Correlation matrix

Compute and visualise the correlation matrix between `log_Weekly_Sales`, `log_CPI`, `Holiday_Flag`, `Temperature`, `Fuel_Price`, and `Unemployment`.

**Hint:** Use `sns.heatmap()` with `annot=True` for a clear visualisation.

In [4]:
cols = ['log_Weekly_Sales', 'log_CPI', 'Holiday_Flag', 'Temperature', 'Fuel_Price', 'Unemployment']
sns.heatmap(data[cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0); plt.title('Correlation matrix'); plt.show()

### Task 1.4: Scatterplots of predictors vs log sales

Create scatterplots of each predictor (`log_CPI`, `Temperature`, `Fuel_Price`, `Unemployment`) against `log_Weekly_Sales`, coloured by `Holiday_Flag`.

**Hint:** Use a 2x2 subplot grid for a compact layout.

In [5]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, c in zip(axes.flat, ['log_CPI', 'Temperature', 'Fuel_Price', 'Unemployment']): sns.scatterplot(data=data, x=c, y='log_Weekly_Sales', hue='Holiday_Flag', s=8, alpha=.5, ax=ax)
plt.tight_layout(); plt.show()

### Discussion

**Answer the following:**
- Which predictors show the strongest correlation with log weekly sales?
- Do you notice any patterns in the scatterplots (e.g., banding, clusters)? What might explain them?
- Are there any pairs of predictors that are highly correlated with each other? Why could this be a problem?

- **Strongest correlations with log sales** are all weak: `Temperature` (−0.09), `Unemployment` (−0.08), `log_CPI` (−0.07). `Holiday_Flag` (+0.03) and `Fuel_Price` (+0.02) are close to zero.
- **Patterns:** there is strong horizontal **banding** in log sales. Each band is one store with its own baseline size. `log_CPI` forms vertical **clusters** because CPI is regional, so stores in the same region share a CPI series. Holiday weeks sit slightly higher.
- **Correlated predictors:** the strongest pair is `log_CPI`–`Unemployment` (−0.30), then `log_CPI`–`Fuel_Price` (−0.17). These are moderate, not severe, but correlated predictors make coefficients unstable and harder to interpret (multicollinearity).

---
## 2. Baseline Log-Log Regression

### Task 2.1: Fit a full model with all predictors

Fit an OLS regression of `log_Weekly_Sales` on all five predictors: `Holiday_Flag`, `Temperature`, `Fuel_Price`, `Unemployment`, `log_CPI`.

**Hint:** Use `sm.OLS()` with `sm.add_constant()`. Print the full summary.

In [6]:
y = data.log_Weekly_Sales
fit = lambda X: sm.OLS(y, sm.add_constant(data[X])).fit()
m_full = fit(['Holiday_Flag', 'Temperature', 'Fuel_Price', 'Unemployment', 'log_CPI']); print(m_full.summary())

                            OLS Regression Results                            
Dep. Variable:       log_Weekly_Sales   R-squared:                       0.021
Model:                            OLS   Adj. R-squared:                  0.021
Method:                 Least Squares   F-statistic:                     27.95
Date:                Tue, 20 Oct 2026   Prob (F-statistic):           4.11e-28
Time:                        12:18:30   Log-Likelihood:                -5647.3
No. Observations:                6435   AIC:                         1.131e+04
Df Residuals:                    6429   BIC:                         1.135e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const           15.1877      0.204     74.449   

### Discussion

**Answer the following:**
- What is the R-squared? Is it high or low? Why might this be the case?
- Which predictors are statistically significant (p < 0.05) and which are not?
- What is the coefficient on `log_CPI`? How do you interpret it as an elasticity?
- Look at the Durbin-Watson statistic. What value is it? What does it suggest?

- **R² = 0.021**, very low. The model ignores the main driver of sales: differences between stores (size, location).
- **Significant:** `Temperature`, `Unemployment`, `log_CPI` (all p < 0.001). **Not significant:** `Fuel_Price` (p = 0.385) and `Holiday_Flag` (p = 0.145).
- **log_CPI = −0.226:** a 1% rise in CPI is associated with a **0.23% fall** in weekly sales.
- **Durbin-Watson = 0.058**, close to 0. This means very strong positive autocorrelation: consecutive weeks of the same store are almost identical.

### Task 2.2: Backward selection — drop non-significant predictors

Iteratively remove the least significant predictor and re-fit the model until all remaining predictors are significant.

**Steps:**
1. Identify the predictor with the highest p-value above 0.05
2. Remove it and re-fit
3. Repeat until all predictors have p < 0.05

In [7]:
X = ['Holiday_Flag', 'Temperature', 'Unemployment', 'log_CPI']
m2 = fit(X); print(m2.summary())

                            OLS Regression Results                            
Dep. Variable:       log_Weekly_Sales   R-squared:                       0.021
Model:                            OLS   Adj. R-squared:                  0.021
Method:                 Least Squares   F-statistic:                     34.75
Date:                Tue, 20 Oct 2026   Prob (F-statistic):           9.40e-29
Time:                        12:18:30   Log-Likelihood:                -5647.7
No. Observations:                6435   AIC:                         1.131e+04
Df Residuals:                    6430   BIC:                         1.134e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const           15.2698      0.181     84.452   

In [8]:
# keep dropping the least significant predictor until all p < 0.05
while (p := fit(X).pvalues.drop('const')).max() > .05: print('Dropping', p.idxmax()); X.remove(p.idxmax())
model = fit(X); print(model.summary())

Dropping Holiday_Flag
                            OLS Regression Results                            
Dep. Variable:       log_Weekly_Sales   R-squared:                       0.021
Model:                            OLS   Adj. R-squared:                  0.020
Method:                 Least Squares   F-statistic:                     45.66
Date:                Tue, 20 Oct 2026   Prob (F-statistic):           3.41e-29
Time:                        12:18:30   Log-Likelihood:                -5648.7
No. Observations:                6435   AIC:                         1.131e+04
Df Residuals:                    6431   BIC:                         1.133e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const           15.2682   

### Discussion

**Answer the following:**
- What is the elasticity of sales with respect to CPI in your final model?
- Is demand elastic or inelastic? What does this mean for Walmart's business?
- How does the unemployment coefficient translate into a business insight?

- **CPI elasticity = −0.231** in the final model (Temperature, Unemployment, log_CPI). `Fuel_Price` was dropped first, then `Holiday_Flag` (p = 0.157).
- **Inelastic** (|−0.23| < 1). A 1% rise in prices lowers sales by only about 0.23%. As a low-price retailer, Walmart keeps its customers when prices rise, and shoppers trading down to it partly offset losses.
- **Unemployment = −0.0315:** +1 pp unemployment means about **3.1% lower** weekly sales. Sales depend on local labour markets, so plan inventory and promotions by region.

---
## 3. Model Diagnostics

### Task 3.1: Multicollinearity — Variance Inflation Factors

Compute VIF for each predictor in your final model. A VIF > 10 indicates problematic multicollinearity.

**Hint:** Use `variance_inflation_factor` from `statsmodels.stats.outliers_influence`.

In [9]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

Xc = sm.add_constant(data[X])
print(pd.Series([variance_inflation_factor(Xc.values, i) for i in range(1, Xc.shape[1])], index=X, name='VIF').round(2))

Temperature     1.05
Unemployment    1.13
log_CPI         1.15
Name: VIF, dtype: float64


### Task 3.2: Residual diagnostics

Create two plots:
1. **Fitted values vs residuals** — look for patterns (should be random scatter)
2. **Histogram of residuals** — check for normality

Also report the Durbin-Watson statistic from the model.

In [10]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(model.fittedvalues, model.resid, s=5, alpha=.4); ax[0].axhline(0, c='r'); ax[0].set(xlabel='Fitted', ylabel='Residuals', title='Fitted vs residuals')
ax[1].hist(model.resid, bins=50); ax[1].set(title='Residual distribution'); plt.show()
print(f'Durbin-Watson: {sm.stats.durbin_watson(model.resid):.3f}')

Durbin-Watson: 0.058


### Discussion

**Answer the following:**
- Are the VIF values concerning? What causes multicollinearity in this dataset?
- What pattern do you see in the fitted-vs-residuals plot? What does it suggest?
- What does the Durbin-Watson statistic tell you? Why is autocorrelation expected in this data?

- **VIFs are low (≈1.05–1.15)**, so multicollinearity is not a concern in the final model. The strongest link is CPI–Unemployment (r = −0.30), since both are regional macro series. (The lecture's VIF > 10 comes from the older, uncentered VIF without a constant.)
- **Fitted vs residuals:** clear **horizontal stripes** instead of a random cloud. These are omitted store effects: each store's baseline level ends up in the residual.
- **DW = 0.058** means strong positive autocorrelation. This is expected: the data stacks 45 weekly time series, and each store's sales persist from week to week (plus seasonality).

---
## 4. Store Fixed Effects

The low R-squared and extreme Durbin-Watson are caused by **unobserved store heterogeneity** — each store has a different baseline sales level driven by size, location, foot traffic, etc. By adding a dummy variable for each store, we absorb these differences and isolate the **within-store** variation in sales.

### Task 4.1: Fit the model with store fixed effects

Add `C(Store)` to the regression formula using `statsmodels.formula.api`. This creates 44 dummy variables (one per store, with one as reference).

**Hint:** Use `smf.ols()` with a formula string like `'y ~ x1 + x2 + C(Store)'`. Print only the key coefficients (not the 44 store dummies).

In [11]:
import statsmodels.formula.api as smf

fe = smf.ols('log_Weekly_Sales ~ log_CPI + Temperature + Unemployment + Holiday_Flag + C(Store)', data).fit()
print(f'R2 = {fe.rsquared:.4f} | Adj. R2 = {fe.rsquared_adj:.4f}')
print(fe.summary2().tables[1].loc[['Intercept', 'log_CPI', 'Temperature', 'Unemployment', 'Holiday_Flag']])

R2 = 0.9582 | Adj. R2 = 0.9578
                  Coef.  Std.Err.          t          P>|t|     [0.025  \
Intercept     15.030437  0.663407  22.656429  2.245822e-109  13.729936   
log_CPI       -0.115645  0.120094  -0.962954   3.356071e-01  -0.351069   
Temperature   -0.000366  0.000099  -3.710536   2.085786e-04  -0.000560   
Unemployment  -0.017772  0.003399  -5.227968   1.768261e-07  -0.024436   
Holiday_Flag   0.058853  0.006006   9.799029   1.637142e-22   0.047079   

                 0.975]  
Intercept     16.330937  
log_CPI        0.119779  
Temperature   -0.000173  
Unemployment  -0.011108  
Holiday_Flag   0.070627  


### Task 4.2: Compare diagnostics — pooled vs fixed effects

Create a 2x2 subplot comparing residual plots from the pooled model (Section 2) and the fixed effects model:
- **Top row:** Pooled model — fitted vs residuals + histogram
- **Bottom row:** FE model — fitted vs residuals + histogram

Include R-squared and Durbin-Watson in each subplot title.

**Hint:** Use `durbin_watson()` from `statsmodels.stats.stattools`.

In [12]:
from statsmodels.stats.stattools import durbin_watson

fig, ax = plt.subplots(2, 2, figsize=(14, 10))
for r, (n, m) in enumerate([('Pooled', model), ('Store FE', fe)]):
    ax[r, 0].scatter(m.fittedvalues, m.resid, s=5, alpha=.4); ax[r, 0].axhline(0, c='r'); ax[r, 0].set(xlabel='Fitted', ylabel='Residuals'); ax[r, 1].hist(m.resid, bins=50)
    for a in ax[r]: a.set_title(f'{n}: R²={m.rsquared:.3f}, DW={durbin_watson(m.resid):.2f}')
plt.tight_layout(); plt.show()

### Task 4.3: Single-store elasticity model

To build intuition for the within-store effect, zoom into **one specific store** and fit the same log-log model (`log_Weekly_Sales ~ log_CPI + Temperature + Unemployment`). This removes the panel structure entirely.

**Steps:**
1. Pick a store (e.g., the one with the highest sales variability)
2. Filter the data to that store only
3. Fit an OLS log-log model with `sm.add_constant()`
4. Print the summary and compare the CPI elasticity to the pooled and FE models

**Hint:** Use `data.groupby('Store')['log_Weekly_Sales'].std().idxmax()` to find the most variable store.

In [13]:
s = data.groupby('Store')['log_Weekly_Sales'].std().idxmax(); d = data[data.Store == s]
ms = sm.OLS(d.log_Weekly_Sales, sm.add_constant(d[['log_CPI', 'Temperature', 'Unemployment']])).fit()
print(f'Store {s}'); print(ms.summary()); print(f"CPI elasticity (store {s}): {ms.params['log_CPI']:.3f}")
el = pd.Series({'Pooled': model.params['log_CPI'], f'Store {s}': ms.params['log_CPI'], 'Store FE': fe.params['log_CPI']})
el.plot.bar(rot=0, color=['grey', 'orange', 'steelblue'], title='CPI elasticity by approach'); plt.axhline(0, c='k'); plt.show()

Store 35
                            OLS Regression Results                            
Dep. Variable:       log_Weekly_Sales   R-squared:                       0.386
Model:                            OLS   Adj. R-squared:                  0.373
Method:                 Least Squares   F-statistic:                     29.17
Date:                Tue, 20 Oct 2026   Prob (F-statistic):           1.09e-14
Time:                        12:18:30   Log-Likelihood:                 58.585
No. Observations:                 143   AIC:                            -109.2
Df Residuals:                     139   BIC:                            -97.32
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const           29.9805      4.399     

### Discussion

**Answer the following:**
- How did R-squared and Durbin-Watson change with store fixed effects? Why?
- Did the residual banding disappear? What does this confirm?
- How does the single-store elasticity compare to the pooled and FE estimates? What does this tell you?
- Why is the fixed effects model more trustworthy than the pooled model for estimating elasticities?

- **R² rose from 0.021 to 0.958 and DW from 0.058 to 1.04.** Store dummies absorb each store's baseline level, which was almost all of the variance and of the residual persistence. DW is still below 2 because of remaining seasonality.
- **The banding disappears**, which confirms the stripes came from unobserved store heterogeneity.
- **Single store (Store 35):** elasticity is **−4.14**, while pooled is −0.23 and FE is −0.12 (p = 0.34, not significant). Unemployment even turns positive (+0.46). With only 143 weeks, CPI and unemployment are basically time trends, so the single-store estimate picks up that store's sales trend and is unreliable. Elasticities also vary a lot across stores.
- **FE is more trustworthy** because it compares each store with itself. That removes bias from time-invariant confounders (size, location, demographics) that are correlated with regional CPI. It is also estimated on all 6,435 observations, unlike the single-store model.

---
## 5. Interpretation and Business Implications

### Task 5.1: Summarise the elasticity estimates

Create a summary comparing the pooled model (Section 2) and the store fixed effects model (Section 4). Include:
- CPI elasticity estimate and interpretation
- Unemployment effect and interpretation
- R-squared and Durbin-Watson for both models
- Whether demand is elastic or inelastic

In [14]:
res = [('Pooled', model), ('Store FE', fe)]
print(pd.DataFrame({n: {'CPI elasticity': r.params['log_CPI'], 'Unemployment': r.params['Unemployment'], 'R2': r.rsquared, 'DW': durbin_watson(r.resid)} for n, r in res}).round(4))
for n, r in res:
    e, u = r.params['log_CPI'], r.params['Unemployment']
    print(f"{n}: +1% CPI -> {e:+.2f}% sales ({'elastic' if abs(e) > 1 else 'inelastic'}); +1pp unemployment -> {100 * np.expm1(u):+.2f}% sales")

                Pooled  Store FE
CPI elasticity -0.2312   -0.1156
Unemployment   -0.0315   -0.0178
R2              0.0209    0.9582
DW              0.0580    1.0412
Pooled: +1% CPI -> -0.23% sales (inelastic); +1pp unemployment -> -3.11% sales
Store FE: +1% CPI -> -0.12% sales (inelastic); +1pp unemployment -> -1.76% sales


### Discussion

**Answer the following:**
- What is Walmart's estimated price elasticity? Is demand elastic or inelastic? Does this make business sense given Walmart's market positioning?
- Which macroeconomic factor has the largest practical impact on Walmart sales?
- What are the key limitations of this analysis? What data or methods would improve it?
- If you were advising Walmart's pricing team, what recommendations would you make based on these results?

- **Elasticity:** FE estimate **−0.12** (pooled −0.23). Demand is **highly inelastic**, and in the FE model the effect is not even statistically significant. This fits Walmart's "Everyday Low Prices" positioning: its goods are necessities, and customers trade down to Walmart when prices rise.
- **Largest practical impact: Unemployment.** +1 pp means −1.8% (FE) to −3.1% (pooled) in sales, so a 3 pp regional swing is about −5% to −9%. Holidays add about +6% (FE). Temperature is minor.
- **Limitations:** CPI is a macro index, not Walmart's own shelf prices, so this is not a true own-price elasticity. Omitted factors include promotions, competitors and product mix. Residual autocorrelation and seasonality remain (DW = 1.04), there may be endogeneity, and there are only 45 stores over about 3 years. **Improvements:** SKU-level price and quantity data, week fixed effects, clustered or HAC standard errors, lags/seasonal terms, and IV or price experiments.
- **Recommendations:** price increases in line with inflation should barely hurt volume. Target value messaging and inventory at high-unemployment regions. Plan stock for holiday weeks. Validate elasticity with controlled store-level price tests.

---
## 6. Regularised Regression: Ridge and Lasso

In Section 3 we found high VIF values (all > 10), meaning our predictors are highly multicollinear. When predictors are correlated, OLS coefficients become **unstable** — small changes in the data can produce large swings in the estimates.

**Regularisation** adds a penalty to the regression that shrinks coefficients toward zero:

| Method | Penalty | Effect |
|--------|---------|--------|
| **Ridge** (L2) | $\lambda \sum \beta_j^2$ | Shrinks all coefficients — stabilises estimates but keeps all predictors |
| **Lasso** (L1) | $\lambda \sum |\beta_j|$ | Shrinks some coefficients to **exactly zero** — performs variable selection |

**Why does this matter for elasticity?** If multicollinearity inflates the CPI coefficient in the pooled model, regularisation will pull it toward a more stable (and possibly more accurate) estimate. If a predictor is truly irrelevant, Lasso will zero it out.

### Task 6.1: Fit Ridge and Lasso on the pooled model

Use cross-validation to find the optimal regularisation strength $\alpha$ for each method. Compare the coefficients to OLS.

**Steps:**
1. Standardise the predictors using `StandardScaler` (essential so penalties are comparable across features)
2. Fit `RidgeCV` and `LassoCV` from `sklearn.linear_model` with `cv=10`
3. Convert the standardised coefficients back to the original scale by dividing by `scaler.scale_`
4. Print a comparison table: OLS vs Ridge vs Lasso coefficients

**Hint:** Use `alphas = np.logspace(-3, 3, 100)` as the search grid.

In [15]:
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

feats, alphas = ['log_CPI', 'Temperature', 'Unemployment'], np.logspace(-3, 3, 100)
scaler = StandardScaler(); Z = scaler.fit_transform(data[feats])
kf = KFold(10, shuffle=True, random_state=42)  # shuffle: data is sorted by store, unshuffled folds would hold out whole stores
ridge, lasso = RidgeCV(alphas=alphas, cv=kf).fit(Z, y), LassoCV(alphas=alphas, cv=kf, max_iter=10000).fit(Z, y)
coefs = pd.DataFrame({'OLS': model.params.reindex(feats), 'Ridge': ridge.coef_ / scaler.scale_, 'Lasso': lasso.coef_ / scaler.scale_}, index=feats)
print(f'Best alpha: Ridge = {ridge.alpha_:.4g}, Lasso = {lasso.alpha_:.4g}'); print(coefs.round(5))

Best alpha: Ridge = 141.7, Lasso = 0.001
                  OLS    Ridge    Lasso
log_CPI      -0.23125 -0.22440 -0.22569
Temperature  -0.00215 -0.00213 -0.00211
Unemployment -0.03154 -0.03064 -0.03084


### Task 6.2: Visualise how regularisation shrinks coefficients

Plot the **coefficient paths** for Ridge and Lasso as regularisation strength increases. Use a 1x2 subplot grid (Ridge left, Lasso right).

**Steps:**
1. Loop over `alphas = np.logspace(-3, 3, 200)`, fitting Ridge and Lasso at each alpha
2. Store the standardised coefficients
3. Plot each predictor's coefficient as a line against alpha (log scale on x-axis)
4. Add a vertical line at the best alpha from cross-validation

**Hint:** Use `Ridge(alpha=a)` and `Lasso(alpha=a, max_iter=10000)` from `sklearn.linear_model`.

In [16]:
from sklearn.linear_model import Ridge, Lasso

A = np.logspace(-3, 3, 200)
paths = {n: np.array([M(alpha=a, **kw).fit(Z, y).coef_ for a in A]) for n, M, kw in [('Ridge', Ridge, {}), ('Lasso', Lasso, {'max_iter': 10000})]}
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for a, (n, P), best in zip(ax, paths.items(), [ridge.alpha_, lasso.alpha_]):
    for j, f in enumerate(feats): a.plot(A, P[:, j], label=f)
    a.axvline(best, ls='--', c='k', label='CV alpha'); a.set(xscale='log', xlabel='alpha', ylabel='Standardised coef', title=f'{n} coefficient path'); a.legend()
plt.tight_layout(); plt.show()

### Task 6.3: Compare OLS, Ridge, and Lasso elasticity estimates

Create a bar chart comparing the CPI elasticity across all methods: OLS (pooled), Ridge (pooled), Lasso (pooled), and OLS (store FE). Also check which predictors Lasso zeroed out.

In [17]:
comp = pd.DataFrame({'OLS (pooled)': model.params, 'Ridge': coefs.Ridge, 'Lasso': coefs.Lasso, 'OLS (Store FE)': fe.params}).loc[feats]
comp.loc['log_CPI'].plot.bar(rot=0, title='CPI elasticity by method'); plt.axhline(0, c='k'); plt.show()
print('Unemployment coefficient:'); print(comp.loc['Unemployment'].round(5))
print('Lasso zeroed out:', coefs.index[coefs.Lasso == 0].tolist() or 'none')

Unemployment coefficient:
OLS (pooled)     -0.03154
Ridge            -0.03064
Lasso            -0.03084
OLS (Store FE)   -0.01777
Name: Unemployment, dtype: float64
Lasso zeroed out: none


### Discussion

**Answer the following:**
- How do the Ridge and Lasso CPI elasticity estimates compare to OLS? Are they larger, smaller, or similar?
- Did Lasso zero out any predictors? Does this agree with the backward selection results from Section 2?
- Looking at the coefficient path plots, which predictor is the most robust (last to be zeroed by Lasso)?
- Regularisation and fixed effects address different problems. What does each one fix? Could you combine them?
- If you had to advise Walmart's pricing team, which model's elasticity estimate would you trust most? Why?

- **Ridge/Lasso vs OLS:** with shuffled 10-fold CV, the chosen penalties are small (Ridge α ≈ 142, Lasso α = 0.001). Both give **almost the same CPI elasticity as OLS** (about −0.224 vs −0.231). With 6,435 rows and weakly correlated predictors (VIF ≈ 1.1), there is little instability to fix.
- **Lasso zeroed nothing** at the CV alpha, which agrees with backward selection (all three predictors significant). Note: unshuffled CV on store-sorted data picks α = 1000 and zeroes everything, so shuffling matters.
- **Most robust on the Lasso path:** `Temperature` is the last to reach zero (it survives to α ≈ 0.05). `Unemployment` and `log_CPI` vanish around α ≈ 0.04–0.05.
- **Different problems:** regularisation reduces **variance** from multicollinearity, while fixed effects remove **bias** from omitted store heterogeneity. They can be combined: Ridge/Lasso with store dummies, or on within-store demeaned data.
- **Most trusted: store FE (−0.12).** Regularisation cannot fix omitted-variable bias, and FE identifies the elasticity from within-store variation. Overall message: demand is inelastic, with |ε| well below 1.